# LangChain & Ollama 기본 기능 입문

**LangChain과 Ollama 각각의 핵심 기능 위주로** 연습

**구성**
- Part A. LangChain 기본 (PromptTemplate, LLM Wrapper, Chain, Output Parser, Memory)
- Part B. Ollama 기본 (설치, 모델 실행, REST API, Modelfile)
- Part C. 두 가지를 합치기 (가장 단순한 LangChain+Ollama 체인)


---
## 1. LangChain 기본 기능


### A. 환경 설정


In [1]:
!pip install -q --upgrade langchain langchain-core langchain-community langchain-classic langchain-huggingface


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 4.1 MB/s eta 0:00:00a 0:00:01


In [2]:
import langchain
print("langchain 버전:", langchain.__version__)


langchain 버전: 1.3.10


### B. PromptTemplate — 동적 프롬프트 만들기

`{변수}` 형태의 placeholder를 채워서 프롬프트를 동적으로 생성

In [3]:
from langchain_core.prompts import PromptTemplate

# 가장 단순한 형태
template = PromptTemplate.from_template("당신은 {role}입니다. 다음 질문에 답하세요: {question}")

filled_prompt = template.format(role="요리 전문가", question="라면을 더 맛있게 끓이는 방법은?")
print(filled_prompt)


당신은 요리 전문가입니다. 다음 질문에 답하세요: 라면을 더 맛있게 끓이는 방법은?


In [4]:
# 변수가 여러 개인 경우 - input_variables를 명시적으로 정의할 수도 있음
template2 = PromptTemplate(
    input_variables=["topic", "tone"],
    template="{topic}에 대해 {tone} 말투로 3문장 설명해줘.",
)

print(template2.format(topic="인공지능", tone="친근한"))
print()
print(template2.format(topic="인공지능", tone="격식 있는"))


인공지능에 대해 친근한 말투로 3문장 설명해줘.

인공지능에 대해 격식 있는 말투로 3문장 설명해줘.


**핵심 포인트**
- PromptTemplate은 prompt 작성을 '템플릿'으로 분리해서, 변수만 바꿔가며 재사용할 수 있음
- 도메인 적용 시: `{context}`, `{question}` 같은 변수를 두면 RAG 체인의 핵심 구성요소


### C. LLM Wrapper — 여러 LLM을 동일한 인터페이스로 다루기

OpenAI, HuggingFace, Ollama 등 백엔드가 달라도 LangChain에서는 동일한 방식(`.invoke()`)으로 호출


In [5]:
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# 가벼운 한국어 생성 모델을 HuggingFace Wrapper로 감싸기
gen_pipeline = pipeline(
    "text-generation",
    model="skt/kogpt2-base-v2",
    max_new_tokens=50,
    temperature=0.5,
    do_sample=True,
)
hf_llm = HuggingFacePipeline(pipeline=gen_pipeline)

# 동일한 .invoke() 인터페이스로 호출
response = hf_llm.invoke("오늘 날씨가")
print(response)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT

오늘 날씨가


**핵심 포인트**
- `hf_llm.invoke(...)`:  이 `hf_llm` 자리에 Ollama LLM을 그대로 끼워 넣어도 코드 구조가 동일하게 작동


### D. Chain — 컴포넌트 연결하기 (LCEL)

LangChain Expression Language(LCEL)로 `prompt | llm` 형태로 파이프처럼 연결


In [6]:
simple_chain = template | hf_llm

result = simple_chain.invoke({"role": "여행 가이드", "question": "제주도 여행 1박2일 코스 추천해줘"})
print(result)


[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


당신은 여행 가이드입니다. 다음 질문에 답하세요: 제주도 여행 1박2일 코스 추천해줘n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�n�


**핵심 포인트**
- `template | hf_llm`처럼 `|` 연산자로 컴포넌트를 이어 붙임
- 파이프라인이 길어지면 `prompt | llm | output_parser | ...` 식으로 계속 이어 붙일 수 있음


### E. Output Parser — LLM 출력을 구조화하기

LLM의 텍스트 출력을 JSON 등 구조화된 형태로 변환


In [8]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import BaseModel, Field

# 가장 단순한 파서: 텍스트만 깔끔하게 추출
str_parser = StrOutputParser()

chain_with_parser = template | hf_llm | str_parser
result = chain_with_parser.invoke({"role": "영양사", "question": "아침 식사로 좋은 메뉴는?"})
print(type(result), "->", result)


[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<class 'str'> -> 당신은 영양사입니다. 다음 질문에 답하세요: 아침 식사로 좋은 메뉴는?��.��ː��.��ː�.��.�!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


In [9]:
# 구조화 출력이 필요한 경우 - Pydantic 모델로 스키마 정의
class RecipeIdea(BaseModel):
    dish_name: str = Field(description="요리 이름")
    main_ingredients: list = Field(description="주재료 목록")

# 참고: 실제로 안정적인 JSON 구조화 출력을 받으려면 충분히 큰 모델이 필요합니다.
# kogpt2 같은 경량 모델은 JSON 형식을 안정적으로 따르지 못할 수 있어, 이 셀은 설명 목적입니다.
# Ollama의 더 큰 모델(Part C 이후)에서 시도하면 훨씬 안정적으로 동작합니다.

print("JsonOutputParser는 Part C에서 Ollama 모델과 함께 다시 시도합니다.")


JsonOutputParser는 Part C에서 Ollama 모델과 함께 다시 시도합니다.


**핵심 포인트**
- `StrOutputParser`: 가장 기본, 텍스트만 깔끔하게 추출
- `JsonOutputParser`: LLM이 JSON 형식으로 답하도록 유도하고 자동 파싱 — 큰 모델일수록 안정적


### F. Memory — 대화 이력 유지하기

멀티턴 대화에서 이전 대화 내용을 기억


In [10]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 세션별 대화 이력 저장소 (메모리 - 실제 서비스에서는 DB 등으로 대체)
session_store = {}

def get_session_history(session_id):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

chat_chain = chat_prompt | hf_llm

chain_with_memory = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user-001"}}

# 첫 번째 턴
r1 = chain_with_memory.invoke({"input": "내 이름은 철수야"}, config=config)
print("[턴1]", r1)

# 두 번째 턴 - 이전 대화를 기억하는지 확인 (경량모델 한계로 완벽하지 않을 수 있음)
r2 = chain_with_memory.invoke({"input": "내 이름이 뭐라고 했지?"}, config=config)
print("[턴2]", r2)


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[턴1] System: 당신은 친절한 어시스턴트입니다.
Human: 내 이름은 철수야ǔ.Human:���ǔ.Human:��nǐ.Human:�젉nǐ.Human:���.Human:�nǐ.Human:
[턴2] System: 당신은 친절한 어시스턴트입니다.
Human: 내 이름은 철수야
AI: System: 당신은 친절한 어시스턴트입니다.
Human: 내 이름은 철수야ǔ.Human:���ǔ.Human:��nǐ.Human:�젉nǐ.Human:���.Human:�nǐ.Human:
Human: 내 이름이 뭐라고 했지?Human:��n.Human:��n.Human:��n.Human:��n.Human:��n.Human:��n.Human:��n.H


**핵심 포인트**
- `session_id`별로 대화 이력이 분리 저장 / 여러 사용자가 동시에 써도 섞이지 않음
- 경량 모델(kogpt2)은 이전 대화를 잘 활용하지 못할 수 있으며, 다른 모형에서 확인


---
## 2. Ollama 기본 기능


### A. 설치 (Colab 기준)

`zstd` 압축 도구가 빠져 있으면 설치 스크립트가 실패하므로 먼저 설치 필요


In [11]:
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [12]:
import subprocess
import time

# 기존에 떠있는 프로세스가 있다면 정리 후 재시작
!pkill ollama
time.sleep(2)

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print("Ollama 서버 시작 (PID:", process.pid, ")")


Ollama 서버 시작 (PID: 78422 )


In [13]:
!ollama --version
!nvidia-smi  # GPU 런타임 확인 - 출력이 안 되면 런타임 유형을 GPU로 변경하세요


ollama version is 0.30.10
/bin/bash: line 1: nvidia-smi: command not found


### B. 모델 다운로드 및 실행


In [14]:
# 가장 가벼운 모델로 먼저 테스트 (용량/속도 부담 적음)
!ollama pull gemma:2b


In [15]:
!ollama list


NAME                        ID              SIZE      MODIFIED               
gemma:2b                    b50d6c999e59    1.7 GB    Less than a second ago    
gemma:7b-instruct-q4_0      430ed3535049    5.2 GB    2 hours ago               
mistral:7b-instruct-q4_0    b17615239298    4.1 GB    2 hours ago               
llama3:8b-instruct-q4_0     365c0bd3c000    4.7 GB    2 hours ago               


In [ ]:
# CLI에서 직접 실행 (한 번 묻고 답하는 방식)
!ollama run gemma:2b "한국의 수도는 어디야?"

한국의 수도는 서울이입니다. 서울은 한국의 수도로, 국가의 주요 정치적, 경제적
, 사회적, 교육적, 문화적 중심 도시입니다.



### C. REST API로 호출

Python에서는 CLI보다 REST API(`/api/generate`)를 직접 호출하는 방식을 더 많이 사용


In [17]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_ollama(model, prompt, temperature=0.3, keep_alive="5m"):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature},
            "keep_alive": keep_alive,
        },
    )
    result = response.json()
    if "response" not in result:
        print("⚠️ 에러:", result)
        return ""
    return result["response"]

answer = ask_ollama("gemma:2b", "파이썬과 자바의 차이를 한 문장으로 설명해줘")
print(answer)


파이썬은 자바와는 다르게 반복적으로 사용되는 반복 구조를 사용하며, 자바는 반복 구조를 사용하지 않는다.


In [18]:
# 대화형(chat) 엔드포인트 - role 기반 메시지 구조 (system/user/assistant)
CHAT_URL = "http://localhost:11434/api/chat"

def chat_ollama(model, messages, temperature=0.3):
    response = requests.post(
        CHAT_URL,
        json={
            "model": model,
            "messages": messages,
            "stream": False,
            "options": {"temperature": temperature},
        },
    )
    result = response.json()
    return result.get("message", {}).get("content", "")

messages = [
    {"role": "system", "content": "당신은 간결하게 답하는 어시스턴트입니다."},
    {"role": "user", "content": "딥러닝과 머신러닝의 차이는?"},
]

answer = chat_ollama("gemma:2b", messages)
print(answer)


딥러닝과 머신러닝은 모두 데이터를 학습하는 과정이지만, 그 방법과 목표는 크게 다릅니다.

**딥러닝:**

* 데이터를 자동으로 분석하여 새로운 패턴을 발견하는 과정
* 인공지능의 한 종류
* 기계 학습의 필수적 요소

**머신러닝:**

* 데이터를 기반으로 새로운 패턴을 예측하는 과정
* 인공지능의 다른 종류
* 데이터 분석과 기계 학습의 중요한 부분

**결론:**

* 딥러닝은 머신러닝보다 더 복잡하고 다양한 데이터를 사용하며, 더 높은 정확성을 제공합니다.
* 머신러닝은 딥러닝보다 더 간소하고 일반적인 데이터를 사용하며, 딥러닝보다 낮은 정확성을 제공합니다.


**`/api/generate` vs `/api/chat` 차이**
- `/api/generate`: 단순 prompt 한 덩어리를 넣는 방식
- `/api/chat`: system/user/assistant role이 구분된 메시지 리스트 — 멀티턴 대화나 시스템 프롬프트 분리에 유리


### D. Modelfile — 커스텀 모델 만들기

예: "항상 존댓말로 짧게 답하는" 커스텀 모델을 만들기


In [19]:
modelfile_simple = '''
FROM gemma:2b

SYSTEM """
당신은 항상 존댓말을 사용하고, 답변은 2문장을 넘지 않습니다.
"""

PARAMETER temperature 0.3
'''

with open("Modelfile_simple", "w", encoding="utf-8") as f:
    f.write(modelfile_simple)

print(modelfile_simple)



FROM gemma:2b

SYSTEM """
당신은 항상 존댓말을 사용하고, 답변은 2문장을 넘지 않습니다.
"""

PARAMETER temperature 0.3



In [20]:
!ollama create polite-bot -f Modelfile_simple
!ollama list



NAME                        ID              SIZE      MODIFIED               
polite-bot:latest           27716fa8aa35    1.7 GB    Less than a second ago    
gemma:2b                    b50d6c999e59    1.7 GB    52 seconds ago            
gemma:7b-instruct-q4_0      430ed3535049    5.2 GB    2 hours ago               
mistral:7b-instruct-q4_0    b17615239298    4.1 GB    2 hours ago               
llama3:8b-instruct-q4_0     365c0bd3c000    4.7 GB    2 hours ago               


In [21]:
answer = ask_ollama("polite-bot", "오늘 점심 메뉴 추천해줘")
print(answer)


안녕하세요. 오늘 점심 메뉴 추천해 드리겠습니다.


**핵심 포인트**
- `FROM`: 베이스 모델 지정
- `SYSTEM`: 역할/제약을 고정 — 매번 prompt에 반복 입력할 필요 없음
- `PARAMETER`: temperature, top_p 등 생성 옵션 기본값 지정


---
## 3. LangChain + Ollama 합치기

Part A의 LangChain 컴포넌트, Part B의 Ollama 모델을 하나로 연결


In [22]:
!pip install -q langchain-ollama


In [23]:
from langchain_ollama import OllamaLLM

ollama_llm = OllamaLLM(model="gemma:2b", temperature=0.3)

# Part A에서 만든 PromptTemplate을 그대로 재사용 - LLM만 교체
chain_with_ollama = template | ollama_llm

result = chain_with_ollama.invoke({"role": "역사 선생님", "question": "조선시대 과거제도를 간단히 설명해줘"})
print(result)


조선시대 과거제도는 15세기부터 19세기에 걸린 시대적 사건들을 말합니다. 

조선시대 과거제도는 15세기부터 19세기에 걸린 시대적 사건들을 말합니다. 

조선시대 과거제도는 15세기부터 19세기에 걸린 시대적 사건들을 말합니다. 

조선시대 과거제도는 15세기부터 19세기에 걸린 시대적 사건들을 말합니다.


In [24]:
# Output Parser까지 연결
from langchain_core.output_parsers import StrOutputParser

full_chain = template | ollama_llm | StrOutputParser()

result = full_chain.invoke({"role": "IT 컨설턴트", "question": "클라우드와 온프레미스의 차이는?"})
print(result)


**클라우드와 온프레미스는 어떻게 차이나는가?**

**클라우드**

* 클라우드는 네트워크를 중앙에 관리하는 시스템입니다.
* 클라우드는 여러 개의 클라우드 호스팅에 분포되어 있으며, 각 호스팅은 다른 클라우드 호스팅과 연결됩니다.
* 클라우드는 데이터를 공유하고 사용할 수 있는 모든 기기와 사용자에 공개적으로 제공할 수 있습니다.
* 클라우드는 일반적으로 비용이 저렴하고 사용하기 간편합니다.

**온프레미스**

* 온프레미스는 클라우드와 유사한 기능을 제공하는 시스템이지만, 클라우드와는 다르게 중앙 관리가 없습니다.
* 온프레미스는 사용자에게 데이터를 직접 사용할 수 있는 시스템입니다.
* 온프레미스는 일반적으로 비용이 클라우드보다 높지만, 사용자에게 더 많은 기능을 제공할 수 있습니다.
* 온프레미스는 데이터의 보안과 분산을 더 높습니다.


In [25]:
# Memory도 Ollama로 다시 시도 - 더 안정적으로 이전 대화를 기억하는지 확인
chat_chain_ollama = chat_prompt | ollama_llm

chain_with_memory_ollama = RunnableWithMessageHistory(
    chat_chain_ollama,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config2 = {"configurable": {"session_id": "user-002"}}

r1 = chain_with_memory_ollama.invoke({"input": "내 이름은 영희야"}, config=config2)
print("[턴1]", r1)

r2 = chain_with_memory_ollama.invoke({"input": "내 이름이 뭐라고 했지?"}, config=config2)
print("[턴2]", r2)


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


[턴1] Hello! It's a pleasure to meet you, and I'm happy to assist you in any way I can. How may I help you today?
[턴2] Hello! My name is Hyungyi. It's a pleasure to meet you as well. How may I assist you today?


**관찰 포인트**
- 백엔드만 교체: 윗부분의 `template`, `chat_prompt` 코드를 유지하고, LLM만 `hf_llm` → `ollama_llm`으로 교체
- Gemma 2B 같은 비교적 작은 모델도 kogpt2보다 안정적으로 이전 대화를 기억하는 경향
